<a href="https://colab.research.google.com/github/seo104/26-2-ESAA/blob/main/ESAA_12%EC%A3%BC%EC%B0%A8_%EC%88%98%EC%83%81%EC%9E%91_%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#수상작 리뷰
---
##[와인 품질(Quality) 분류 경진대회]
https://dacon.io/competitions/open/235610/overview/description

---
###데이터
train.csv / test.csv
- index 구분자
- quality 품질
- fixed acidity 산도
- volatile acidity 휘발성산
- citric acid 시트르산
- residual sugar 잔당 : 발효 후 와인 속에 남아있는 당분
- chlorides 염화물
- free sulfur dioxide 독립 이산화황
- total sulfur dioxide 총 이산화황
- density 밀도
- pH 수소이온농도
- sulphates 황산염
- alcohol 도수
- type 종류

sample_submission.csv

---
###코드 흐름
1. 데이터 불러오기 및 초기 확인
    - 필요한 라이브러리 불러오기 및 시각화 스타일 설정
    - train.csv, test.csv, sample_submission.csv 데이터 불러오기
    - index 컬럼 제거 후 데이터의 크기, 자료형, 기초 통계량 확인
2. 데이터 탐색 및 시각화(EDA)
    - pandas profiling을 활용하여 변수별 분포, 결측값, 상관관계 등 데이터 전반 확인
    - 품질 등급별 데이터 개수를 확인하고 분포 시각화
    - 품질 등급에 따른 각 수치형 변수의 분포 비교
    - 상관관계 히트맵을 통해 변수 간 관계 확인
3. 데이터 전처리
    - 수치형 변수의 단위와 스케일 차이를 줄이기 위해 StandardScaler로 표준화
    - 와인 종류(type)를 숫자형으로 변환
    - quality를 타깃 변수(y), 나머지 변수를 입력 변수(X)로 설정
4. 모델 학습 및 평가
    - 데이터를 학습용과 검증용으로 8:2 분리
    - RandomForestClassifier 모델 학습
    - 정확도(Testing Score)와 혼동행렬을 통해 분류 결과 확인
5. 테스트 데이터 예측 및 제출
    - 학습 데이터와 동일하게 test 데이터의 index 제거 및 전처리
    - 학습한 Random Forest 모델로 품질 등급 예측
    - 예측 결과를 sample_submission.csv에 입력하고 submission.csv로 저장
6. 추가 내용 : 앙상블 학습
    - Random Forest의 부트스트랩 샘플 및 특성 선택 방식 설명
    - Extra Trees와 Random Forest의 차이점 소개
---
###주요 코드 분석
① 품질 등급에 따른 수치형 변수 분포 비교

In [ ]:
numerical_columns = train.select_dtypes(exclude='object').columns.tolist()
numerical_columns.remove('quality')

def show_dist_plot(df, columns):
    for column in columns:
        f, ax = plt.subplots(1,2,figsize=(16,4))
        sns.stripplot(x=df['quality'],y=df[column], ax=ax[0],hue=df['quality'])
        sns.violinplot(data=df, x='quality', y=column, ax=ax[1])

show_dist_plot(train, numerical_columns)

- select_dtypes(exclude='object')로 문자형이 아닌 수치형 변수만 선택하고, 품질 등급인 quality는 제외
- 각 변수에 대해 Strip plot과 Violin plot을 함께 그려 품질 등급별 분포를 비교
- Strip plot은 개별 데이터의 분포를, Violin plot은 값이 밀집된 구간과 전체 분포 형태를 확인하는 데 활용

② 변수 간 상관관계 확인

In [ ]:
plt.figure(figsize=(18,8))
corr = train.corr()
sns.heatmap(corr, annot=True, square=False, vmin=-.6, vmax=1.0);

- train.corr()로 수치형 변수 간 상관계수를 계산
- heatmap()으로 변수 간 상관관계를 시각화하고 annot=True로 상관계수 표시
- 변수 간 관계를 파악하여 품질 분석에 참고할 수 있는 정보를 확인
- 단, 상관관계는 변수 간 함께 변화하는 경향을 나타낼 뿐 인과관계를 의미하지 않음

③ StandardScaler를 활용한 표준화 및 와인 종류 변환

In [ ]:
ss = StandardScaler()
train[numerical_columns] = ss.fit_transform(train[numerical_columns])

train['type'] = pd.factorize(train['type'])[0]

- StandardScaler를 이용해 수치형 변수의 스케일을 표준화
- 서로 단위와 값의 범위가 다른 변수들을 비슷한 기준으로 변환
- pd.factorize()를 이용해 Red와 White로 구성된 type 변수를 숫자로 변환하여 모델 학습에 사용할 수 있도록 처리
---
###새롭게 알게 된 점/배운 점
- Strip plot과 Violin plot을 함께 사용하면 개별 데이터의 분포와 전체적인 분포 형태를 동시에 비교할 수 있음.
- StandardScaler를 활용하면 단위와 범위가 다른 수치형 변수들을 표준화할 수 있으며, 문자형 범주 변수도 숫자로 변환해야 머신러닝 모델에 사용할 수 있음.
- Random Forest와 Extra Trees는 여러 결정 트리를 결합하는 앙상블 모델이지만, 부트스트랩 샘플 사용 여부와 노드 분할 방식에 차이가 있음.